# Notebook 4: New Sample Prediction & Recommendations

In this notebook I apply the trained KMeans model to the test dataset (the user's listening history).
After assigning clusters to those songs, I'll recommend new songs from the same clusters.

In [1]:
# Importing libraries
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler

# Load the test dataset from the data folder
df_test = pd.read_csv('../data/recommend - recommend.csv')
print("Test data shape:", df_test.shape)
df_test.head()

Test data shape: (10, 25)


,Unnamed: 0,artist_name,track_name,release_date,genre,lyrics,len,dating,violence,world/life,...,obscene,music,movement/places,light/visual perceptions,family/spiritual,like/girls,sadness,feelings,topic,age
0,76885,godsmack,immune,1998,rock,come world society futher place home land deat...,74,0.000907,0.348191,0.375448,...,0.000907,0.019389,0.000907,0.000907,0.000907,0.000907,0.000907,0.018854,world/life,0.314286
1,65394,dennis brown,second chance,1993,reggae,maybe maybe treat good feel second best girl s...,43,0.001224,0.029943,0.001224,...,0.001224,0.001224,0.001224,0.001224,0.001224,0.056842,0.001224,0.062092,night/time,0.385714
2,10980,the black crowes,sister luck,1990,pop,worry sick eye hurt rest head life outside gir...,54,0.001120,0.482490,0.001120,...,0.001120,0.001120,0.001120,0.078222,0.001120,0.051132,0.031571,0.202862,violence,0.428571
3,842,jerry lee lewis,your cheating heart,1960,pop,cheat heart weep sleep sleep come night cheat ...,25,0.204740,0.002506,0.002506,...,0.002506,0.002506,0.002506,0.002506,0.002506,0.002506,0.474607,0.002506,sadness,0.857143
4,2764,paul anka,eso beso,1966,pop,beso kiss beso kiss know samba bossanova close...,97,0.001170,0.001170,0.001170,...,0.001170,0.001170,0.001170,0.314626,0.001170,0.053731,0.001170,0.001170,romantic,0.771429


## 1. Preprocess the Test Data

I apply the same preprocessing steps as the training data:
- Drop the same non-numeric and identifier columns
- Drop the same highly correlated columns that were dropped during training
- Scale the data

In [2]:
# Save test identifiers
id_cols = ['artist_name', 'track_name', 'release_date', 'genre', 'topic']
# topic may not exist in test — handle gracefully
id_cols_present = [c for c in id_cols if c in df_test.columns]
test_ids = df_test[id_cols_present].copy()

# Drop same columns as training
cols_to_drop = ['Unnamed: 0', 'lyrics', 'artist_name', 'track_name',
                'release_date', 'genre', 'topic']
cols_to_drop_present = [c for c in cols_to_drop if c in df_test.columns]
df_test_clean = df_test.drop(columns=cols_to_drop_present)

print("Test data shape after dropping:", df_test_clean.shape)
print("Columns:", df_test_clean.columns.tolist())

Test data shape after dropping: (10, 18)
Columns: ['len', 'dating', 'violence', 'world/life', 'night/time', 'shake the audience', 'family/gospel', 'romantic', 'communication', 'obscene', 'music', 'movement/places', 'light/visual perceptions', 'family/spiritual', 'like/girls', 'sadness', 'feelings', 'age']


In [3]:
# Load the training cleaned data to get the exact feature set used
df_train_cleaned = pd.read_csv('../data/train_cleaned.csv')
train_features = df_train_cleaned.columns.tolist()
print("Training features used:", train_features)

# Keep only the columns that were used during training
# Some columns (like 'like/girls') may appear in test but not train — drop those
df_test_clean = df_test_clean[train_features]
print("Test data shape after aligning features:", df_test_clean.shape)

Training features used: ['len', 'dating', 'violence', 'world/life', 'night/time', 'shake the audience', 'family/gospel', 'romantic', 'communication', 'obscene', 'music', 'movement/places', 'light/visual perceptions', 'family/spiritual', 'sadness', 'feelings', 'age']
Test data shape after aligning features: (10, 17)


In [4]:
# Load raw training data to fit the scaler correctly
# We need unscaled data so the scaler learns the right mean and std from training
df_train_raw = pd.read_csv('../train.csv')
cols_to_drop_raw = ['Unnamed: 0', 'lyrics', 'artist_name', 'track_name',
                    'release_date', 'genre', 'topic']
df_train_raw = df_train_raw.drop(columns=[c for c in cols_to_drop_raw if c in df_train_raw.columns])
df_train_raw = df_train_raw[train_features]

# Fit on training data, then transform test data — ensures consistent scaling
scaler = StandardScaler()
scaler.fit(df_train_raw)
df_test_scaled = scaler.transform(df_test_clean)
df_test_scaled = pd.DataFrame(df_test_scaled, columns=train_features)

print("Scaled test data shape:", df_test_scaled.shape)
df_test_scaled.head()

Scaled test data shape: (10, 17)


,len,dating,violence,world/life,night/time,shake the audience,family/gospel,romantic,communication,obscene,music,movement/places,light/visual perceptions,family/spiritual,sadness,feelings,age
0,0.023176,-0.385794,1.286387,1.477614,-0.504500,5.115865,-0.384526,-0.450352,-0.691752,-0.531010,-0.329793,-0.507983,-0.537130,-0.455554,-0.709340,-0.169436,-0.419781
1,-0.717904,-0.379749,-0.494964,-0.695418,2.228376,-0.398319,0.390650,0.197620,3.196671,-0.529264,-0.477063,-0.504525,-0.533595,-0.449351,-0.707592,0.433986,-0.149316
2,-0.454940,-0.381738,2.038110,-0.696023,-0.502602,-0.400881,2.301915,-0.448350,-0.689812,-0.529838,-0.477907,-0.505663,0.326225,-0.451392,-0.540063,2.398551,0.012963
3,-1.148208,3.506697,-0.648540,-0.687972,0.647622,-0.366780,-0.346429,-0.435279,-0.677150,-0.522192,-0.466667,-0.490520,-0.519276,-0.424224,1.905675,-0.397588,1.635754
4,0.573009,-0.380788,-0.656022,-0.695734,-0.063159,-0.399657,0.154522,4.645790,-0.689358,-0.529564,-0.477504,-0.505120,2.966110,-0.450417,-0.707893,-0.416242,1.311196


## 2. Apply the Trained KMeans Model

In [5]:
# Load the trained KMeans model saved from notebook 3
with open('../data/kmeans_model.pkl', 'rb') as f:
    kmeans_model = pickle.load(f)

# Predict which cluster each test song belongs to
test_cluster_labels = kmeans_model.predict(df_test_scaled)

# Attach cluster labels to the song identifiers dataframe
test_ids = test_ids.reset_index(drop=True)
test_ids['cluster'] = test_cluster_labels

print("Cluster assignments for the user's songs:")
print(test_ids[['artist_name', 'track_name', 'cluster']].to_string(index=False))

Cluster assignments for the user's songs:
             artist_name           track_name  cluster
                godsmack               immune       10
            dennis brown        second chance        2
        the black crowes          sister luck        8
         jerry lee lewis  your cheating heart        0
               paul anka             eso beso        3
            noro morales             silencio        8
rage against the machine     pistol grip pump        1
                   taste      railway and gun        0
            randy travis messin' with my mind        2
                paramore          playing god        8


## 3. Generate Recommendations

Now I look at which clusters the user's songs belong to,
and find other songs from the training set in those same clusters.
Those are the recommendations!

In [6]:
# Load the training set with cluster labels — this is our recommendation pool
df_train_clusters = pd.read_csv('../data/train_with_clusters.csv')

# Find which clusters the user's songs belong to
user_clusters = test_ids['cluster'].unique()
print("User's songs fall into these clusters:", user_clusters)

# Build a set of the user's track names to avoid recommending what they already have
user_tracks = set(test_ids['track_name'].str.lower())

recommendations = []
for cluster in user_clusters:
    cluster_songs = df_train_clusters[df_train_clusters['cluster'] == cluster]
    # Exclude songs already in the user's listening history
    cluster_songs = cluster_songs[
        ~cluster_songs['track_name'].str.lower().isin(user_tracks)
    ]
    # Sample 3 songs per cluster as recommendations
    sample = cluster_songs[['artist_name', 'track_name', 'genre', 'cluster']].sample(
        min(3, len(cluster_songs)), random_state=42
    )
    recommendations.append(sample)

df_recommendations = pd.concat(recommendations, ignore_index=True)
print("\nRecommended songs for this user:")
print(df_recommendations.to_string(index=False))

User's songs fall into these clusters: [10  2  8  0  3  1]

Recommended songs for this user:
          artist_name                 track_name   genre  cluster
               krokus                heatstrokes   blues       10
             loverboy    working for the weekend     pop       10
         thee oh sees           the experimenter   blues       10
        george benson           unchained melody    jazz        2
        cocteau twins                   aloysius     pop        2
    the kingston trio                    wimoweh    rock        2
                creed           a thousand faces    rock        8
sister rosetta tharpe                    jericho   blues        8
            mel tormé i've got you under my skin    jazz        8
          ringo starr                 photograph     pop        0
        nate millyunz                all my life hip hop        0
            ray price          same old memories country        0
       emmylou harris           pledging my love 

## 4. Save Results

In [7]:
# Save test songs with their assigned cluster labels
test_ids.to_csv('../data/test_with_clusters.csv', index=False)
print("Saved test_with_clusters.csv")

# Save the recommended songs for reporting
df_recommendations.to_csv('../data/recommendations.csv', index=False)
print("Saved recommendations.csv")

print("\nFinal recommendation summary:")
print(df_recommendations[['artist_name', 'track_name', 'genre']].to_string(index=False))

Saved test_with_clusters.csv
Saved recommendations.csv

Final recommendation summary:
          artist_name                 track_name   genre
               krokus                heatstrokes   blues
             loverboy    working for the weekend     pop
         thee oh sees           the experimenter   blues
        george benson           unchained melody    jazz
        cocteau twins                   aloysius     pop
    the kingston trio                    wimoweh    rock
                creed           a thousand faces    rock
sister rosetta tharpe                    jericho   blues
            mel tormé i've got you under my skin    jazz
          ringo starr                 photograph     pop
        nate millyunz                all my life hip hop
            ray price          same old memories country
       emmylou harris           pledging my love country
          aaron shust                    ever be    rock
        marty robbins              forever yours country
  